## Load the environment variables

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

/home/v/Desktop/Projects/HuggingFace/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load a dataset

`load_dataset` downloads the dataset from the Hub and caches it locally.
On subsequent runs it reads from the cache — no re-download needed.

The return value is a `DatasetDict`: a dict-like object with a key per split (`train`, `test`, `validation`).

We'll use **spain-real-estate-open-data** — just out of curiosity, to see what's in the dataset.

In [2]:
ds = load_dataset("Eligemicasa/spain-real-estate-open-data", "municipalities")

## Inspect the dataset

Before touching the data it's worth understanding its shape:
- How many rows per split?
- What columns exist and what types do they have?
- What does a single example look like?

Print the `DatasetDict`, the `features`, and the first example of the training split.

In [3]:
print(ds)
print(ds["train"][0])
print(ds["train"].features)

DatasetDict({
    train: Dataset({
        features: ['municipality_id', 'municipality', 'slug', 'population', 'province', 'province_slug', 'active_listings', 'price_avg'],
        num_rows: 8131
    })
})
{'municipality_id': 4100, 'municipality': 'Abengibre', 'slug': 'abengibre', 'population': 2294, 'province': 'Albacete', 'province_slug': 'albacete', 'active_listings': 0, 'price_avg': None}
{'municipality_id': Value('int64'), 'municipality': Value('string'), 'slug': Value('string'), 'population': Value('int64'), 'province': Value('string'), 'province_slug': Value('string'), 'active_listings': Value('int64'), 'price_avg': Value('float64')}


## Filter rows

`.filter()` keeps only the rows where a function returns `True`.
It's lazy-friendly: it iterates in batches without loading everything into RAM.

Filter the training split to keep only **already hydrated municipalities** (average price not `None`).

In [4]:
priced = ds["train"].filter(lambda x: x["price_avg"] is not None)

print(f"There are {len(priced)} rows in the training split after filtering.")
print("--------------------------------------------------------")
print(priced[0])

There are 13 rows in the training split after filtering.
--------------------------------------------------------
{'municipality_id': 6024, 'municipality': 'Cox', 'slug': 'cox', 'population': 7521, 'province': 'Alicante/Alacant', 'province_slug': 'alicante-alacant', 'active_listings': 24, 'price_avg': 108896.0}


## Map a transformation — tokenization

-- It makes no sense for this dataset, just adding as exercise. --

`.map()` applies a function to every row (or batch of rows) and returns a new dataset with the result added as extra columns.

Here we use it to tokenize the `text` column with `distilbert-base-uncased`.
The tokenizer adds two new columns: `input_ids` and `attention_mask`.

Key arguments to `.map()`:
- `batched=True` — passes a batch (default 1000 rows) at a time instead of one row. Much faster.
- `remove_columns` — drop the raw `text` column after tokenizing so the dataset is model-ready.

Tokenizer arguments to use:
- `truncation=True` — clips sequences longer than the model's max length
- `padding="max_length"` — pads shorter sequences so all tensors have the same shape
- `max_length=128`

In [12]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["province"], truncation=True, padding="max_length", max_length=128)

tokenized = ds.map(tokenize, batched=True)

Map: 100%|██████████| 8131/8131 [00:00<00:00, 47154.51 examples/s]


## Verify the tokenized dataset

Print the first tokenized example to confirm the new columns are there and the shapes look right.
Check that `input_ids` has exactly 128 elements and that `attention_mask` has the same length.

In [13]:
print(tokenized['train'][0])
print(len(tokenized['train'][0]["input_ids"]))
print(len(tokenized['train'][0]["attention_mask"]))

{'municipality_id': 4100, 'municipality': 'Abengibre', 'slug': 'abengibre', 'population': 2294, 'province': 'Albacete', 'province_slug': 'albacete', 'active_listings': 0, 'price_avg': None, 'input_ids': [101, 18255, 3401, 2618, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## Set the format for PyTorch

Before training, the dataset needs to return tensors instead of Python lists.
`.set_format(type="torch", columns=[...])` does this in place — it doesn't change the data, only the return type.

Set the format on the tokenized dataset so that `input_ids`, `attention_mask`, and `label` come back as tensors.

In [15]:
tokenized['train'].set_format(type="torch", columns=["input_ids", "attention_mask"])

print(tokenized['train'][0])

{'input_ids': tensor([  101, 18255,  3401,  2618,   102,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

## Key takeaways

| Concept | What it means |
|---|---|
| `DatasetDict` | Container with one `Dataset` per split (train / test / validation) |
| `.filter(fn)` | Keep only rows where `fn` returns `True` — memory-efficient |
| `.map(fn, batched=True)` | Apply `fn` to every row and add the result as new columns |
| `truncation + padding` | Makes all sequences the same length so they can be batched into tensors |
| `.set_format("torch")` | Switches output type from Python lists to PyTorch tensors |
| `.push_to_hub(...)` | Uploads the dataset to your HF Hub account, versioned and shareable |